In [16]:
import robotic as ry
import numpy as np
import open3d as o3d
     

base_path = "/mnt/c/Users/user/Documents/CS549/grasp_gen/environments/"
base_path_depth = "/mnt/c/Users/user/Documents/CS549/grasp_gen/depth_points/"
base_path_sam = "/mnt/c/Users/user/Documents/CS549/grasp_gen/segmentation_points/"

file_num = 2
env_file_name = f"env_{file_num}.g"
g_file_path = base_path + env_file_name

depth_pts_file_name = f"env_{file_num}_points.npy"
depth_clr_file_name = f"env_{file_num}_colors.npy"
depth_pts_path = base_path_depth + depth_pts_file_name
depth_clr_path = base_path_depth + depth_clr_file_name

sam_pts_file_name = f"env_{file_num}_points.npy"
sam_clr_file_name = f"env_{file_num}_colors.npy"
sam_pts_path = base_path_sam + sam_pts_file_name
sam_clr_path = base_path_sam + sam_clr_file_name
     

C = ry.Config()

C.addFile(g_file_path)

all_frames = C.getFrameNames()
print(all_frames)

#C.view()
     

goal_frames = [f for f in all_frames if "goal" in f]
goal_base_frame = [f for f in goal_frames if "base" in f][0]
goal_prefix = goal_base_frame.replace("_base", "")

goal_cubes = [
    f for f in goal_frames if f.startswith(goal_prefix) and "cube" in f
]
goal_cubes.sort()
print(goal_cubes)
     

goal_pose_frames = [f for f in all_frames if "goal_pose" in f]
goal_pose_base_frame = [f for f in goal_pose_frames if "base" in f][0]
goal_pose_prefix = goal_pose_base_frame.replace("_base", "")

goal_pose_cubes = [
    f for f in goal_pose_frames if f.startswith(goal_pose_prefix) and "cube" in f
]
goal_pose_cubes.sort()

print(goal_pose_cubes)
     

def calculate_centroid(prefix, full_frame_list):
    cubes = [
        f for f in full_frame_list if f.startswith(prefix) and "cube" in f
    ]
    if not cubes:
        return None
    positions = [C.getFrame(f).getPosition() for f in cubes]
    return np.mean(positions, axis=0)
     

goal_centroid = calculate_centroid(goal_prefix, goal_frames)
goal_pose_centroid = calculate_centroid(goal_pose_prefix, goal_pose_frames)

print("Goal Centroid:", goal_centroid)
print("Goal Pose Centroid:", goal_pose_centroid)
     

#pts = np.load(depth_pts_path)
#colors = np.load(depth_clr_path)

pts = np.load(sam_pts_path)
colors = np.load(sam_clr_path)
     

def filter_background_by_coordinates(pts, colors, x_range=(-1.5, 1.5), y_range=(-1.5, 1.5), z_range=(-0.5, 2.0)):
    mask_x = (pts[:, 0] >= x_range[0]) & (pts[:, 0] <= x_range[1])
    mask_y = (pts[:, 1] >= y_range[0]) & (pts[:, 1] <= y_range[1])
    mask_z = (pts[:, 2] >= z_range[0]) & (pts[:, 2] <= z_range[1])
    
    spatial_mask = mask_x & mask_y & mask_z
    
    filtered_pts = pts[spatial_mask]
    filtered_colors = colors[spatial_mask]
    
    print(f"Original points: {pts.shape[0]} | Points kept: {filtered_pts.shape[0]}")
    
    return filtered_pts, filtered_colors
     

x_bounds = (-1.5, 1.5)
y_bounds = (-1.5, 1.5)
z_bounds = (0.65, 2.0)

filtered_pts, filtered_colors = filter_background_by_coordinates(
    pts, colors, x_range=x_bounds, y_range=y_bounds, z_range=z_bounds
)

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(filtered_pts[:, :3])

if filtered_colors.max() > 1.0:
    pcd.colors = o3d.utility.Vector3dVector(filtered_colors[:, :3] / 255.0)
else:
    pcd.colors = o3d.utility.Vector3dVector(filtered_colors[:, :3])

#o3d.visualization.draw_geometries([pcd], window_name="Spatially Cropped Point Cloud")
     

def extract_points_by_centroid(pts, colors, centroid, radius=0.15):
    distances = np.linalg.norm(pts - centroid, axis=1)
    
    mask = distances <= radius
    
    extracted_pts = pts[mask]
    extracted_colors = colors[mask]
    
    print(f"Centroid: {centroid}")
    print(f"Points within {radius}m radius: {extracted_pts.shape[0]}")
    
    return extracted_pts, extracted_colors, distances[mask]
     

radius = 0.15  # Adjust radius as needed

goal_pts, goal_colors, goal_distances = extract_points_by_centroid(
    filtered_pts, filtered_colors, goal_centroid, radius=radius
)

goal_pose_pts, goal_pose_colors, goal_pose_distances = extract_points_by_centroid(
    filtered_pts, filtered_colors, goal_pose_centroid, radius=radius
)
     

goal_pcd = o3d.geometry.PointCloud()
goal_pcd.points = o3d.utility.Vector3dVector(goal_pts[:, :3])
if goal_colors.max() > 1.0:
    goal_pcd.colors = o3d.utility.Vector3dVector(goal_colors[:, :3] / 255.0)
else:
    goal_pcd.colors = o3d.utility.Vector3dVector(goal_colors[:, :3])

goal_pose_pcd = o3d.geometry.PointCloud()
goal_pose_pcd.points = o3d.utility.Vector3dVector(goal_pose_pts[:, :3])
if goal_pose_colors.max() > 1.0:
    goal_pose_pcd.colors = o3d.utility.Vector3dVector(goal_pose_colors[:, :3] / 255.0)
else:
    goal_pose_pcd.colors = o3d.utility.Vector3dVector(goal_pose_colors[:, :3])

# Visualize both point clouds with centroids
geometries = [goal_pcd, goal_pose_pcd]
o3d.visualization.draw_geometries(geometries, window_name="Goal and Goal Pose Point Clouds (Red: Goal, Green: Goal Pose)")          # Comment it out if you don't want to visualize here
     

['world', 'table', 'obj0_base', 'obj0_cube0', 'obj0_joint1', 'obj0_cube1', 'obj0_joint2', 'obj0_cube2', 'obj0_joint3', 'obj0_cube3', 'obj0_joint4', 'obj0_cube4', 'obj0_joint5', 'obj0_cube5', 'obj1_base', 'obj1_cube0', 'obj1_joint1', 'obj1_cube1', 'obj1_joint2', 'obj1_cube2', 'obj1_joint3', 'obj1_cube3', 'obj1_joint4', 'obj1_cube4', 'obj1_joint5', 'obj1_cube5', 'obj2_base', 'obj2_cube0', 'obj2_joint1', 'obj2_cube1', 'obj2_joint2', 'obj2_cube2', 'obj2_joint3', 'obj2_cube3', 'obj2_joint4', 'obj2_cube4', 'obj2_joint5', 'obj2_cube5', 'obj3_base', 'obj3_cube0', 'obj3_joint1', 'obj3_cube1', 'obj3_joint2', 'obj3_cube2', 'obj3_joint3', 'obj3_cube3', 'obj3_joint4', 'obj3_cube4', 'obj3_joint5', 'obj3_cube5', 'obj4_base', 'obj4_cube0', 'obj4_joint1', 'obj4_cube1', 'obj4_joint2', 'obj4_cube2', 'obj4_joint3', 'obj4_cube3', 'obj4_joint4', 'obj4_cube4', 'obj4_joint5', 'obj4_cube5', 'obj5_base', 'obj5_cube0', 'obj5_joint1', 'obj5_cube1', 'obj5_joint2', 'obj5_cube2', 'obj5_joint3', 'obj5_cube3', 'obj5_j

In [17]:
import numpy as np
import open3d as o3d


def align_normals_with_centroid(point_cloud):
    normals = np.asarray(point_cloud.normals)
    pts = np.asarray(point_cloud.points)

    centroid = np.mean(pts, axis=0)
    for i in range(len(normals)):
        to_point = pts[i] - centroid
        if np.dot(normals[i], to_point) < 0:
            normals[i] = -normals[i]

    point_cloud.normals = o3d.utility.Vector3dVector(normals)
    return point_cloud


def reorient_normals_locally(point_cloud):
    normals = np.asarray(point_cloud.normals)
    pts = np.asarray(point_cloud.points)
    kdtree = o3d.geometry.KDTreeFlann(point_cloud)

    for i in range(len(normals)):
        _, idx, _ = kdtree.search_knn_vector_3d(pts[i], 10)
        avg_normal = np.mean(normals[idx], axis=0)
        norm = np.linalg.norm(avg_normal)

        if norm > 1e-12:
            avg_normal = avg_normal / norm
            if np.dot(normals[i], avg_normal) < 0:
                normals[i] = -normals[i]

    point_cloud.normals = o3d.utility.Vector3dVector(normals)
    return point_cloud


# goal_pcd'nin daha önce oluşturulmuş olduğunu varsayıyoruz
# örn: goal_pcd = ...

target_pcd = o3d.geometry.PointCloud(goal_pcd)

points = np.asarray(target_pcd.points)
assert len(points) > 0, "goal_pcd boş geldi"

print("\nLoaded target points shape:", points.shape)
print(points[:5])

colors = None
if target_pcd.has_colors():
    colors = np.asarray(target_pcd.colors)
    print("\nLoaded target colors shape:", colors.shape)
    print(colors[:5])

# raw target object visualization
o3d.visualization.draw_geometries([target_pcd], window_name="Raw Target Object")

# downsample
pcd_tmp = o3d.geometry.PointCloud(target_pcd)
pcd_tmp = pcd_tmp.voxel_down_sample(voxel_size=0.005)

print("\nDownsampled point count:", len(pcd_tmp.points))
if pcd_tmp.has_colors():
    print("Downsampled color count:", len(pcd_tmp.colors))

# estimate normals
pcd_tmp.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.05, max_nn=30)
)

pcd_tmp = align_normals_with_centroid(pcd_tmp)
pcd_tmp = reorient_normals_locally(pcd_tmp)

normals = np.asarray(pcd_tmp.normals)

print("Number of processed points:", len(pcd_tmp.points))
print("Number of normals:", len(normals))

o3d.visualization.draw_geometries(
    [pcd_tmp],
    window_name="Target Object with Normals",
    point_show_normal=True
)


Loaded target points shape: (47316, 3)
[[0.38868743 0.29662988 0.69003296]
 [0.38496793 0.2957     0.69003296]
 [0.3858978  0.2957     0.69003296]
 [0.38775755 0.2957     0.69003296]
 [0.38868743 0.2957     0.69003296]]

Loaded target colors shape: (47316, 3)
[[1. 0. 1.]
 [1. 0. 1.]
 [1. 0. 1.]
 [1. 0. 1.]
 [1. 0. 1.]]

Downsampled point count: 2106
Downsampled color count: 2106
Number of processed points: 2106
Number of normals: 2106


In [18]:
def angle_between(v1, v2):
    v1_u = v1 / np.linalg.norm(v1)
    v2_u = v2 / np.linalg.norm(v2)
    dot_product = np.dot(v1_u, v2_u)
    angle = np.arccos(np.clip(dot_product, -1.0, 1.0))
    return np.degrees(angle)

min_distance = 0.01
max_distance = 0.09
small_angle_threshold = 10  # Max angle 
large_angle_threshold = 160  # Min angle 

best_grasps = []
compute_grasp_points = True
cropped_pcl = np.asarray(pcd_tmp.points)
if compute_grasp_points:
    points = cropped_pcl
    grasp_candidates = []

    centroid = np.mean(points, axis=0)

    for i in range(len(points)):
        for j in range(i + 1, len(points)):
            p1, p2 = points[i], points[j]
            n1, n2 = normals[i], normals[j]

            distance = np.linalg.norm(p1 - p2)
            if min_distance <= distance <= max_distance:
                angle = angle_between(n1, n2)
                if 160 <= angle <= 180:
                    # Compute the epipolar line (direction vector)
                    epiline = p2 - p1
                    epiline /= np.linalg.norm(epiline)

                    # Check normal directions relative to the epipolar line
                    projection_n1 = np.dot(n1, epiline)
                    projection_n2 = np.dot(n2, epiline)

                  
                    if projection_n1 < 0 and projection_n2 > 0:
                        # Calculate the angles between the epipolar line and each normal
                        angle_n1_epiline = angle_between(n1, epiline)
                        angle_n2_epiline = angle_between(n2, epiline)
                        if min(abs(angle_n1_epiline), abs(angle_n2_epiline)) <= small_angle_threshold and max(abs(angle_n1_epiline), abs(angle_n2_epiline)) >= large_angle_threshold:
                            # Calculate closeness to the centroid
                            dist_to_centroid = np.linalg.norm((p1 + p2) / 2 - centroid)
                            grasp_candidates.append((p1, p2, n1, n2, angle, distance, angle_n1_epiline, angle_n2_epiline, dist_to_centroid))
    if grasp_candidates:
        # Select the best grasp considering  closeness to the centroid
        best_grasp = min(grasp_candidates, key=lambda x: x[8] * 1e1)  # Minimize closeness to centroid
        best_grasps.append(best_grasp)
        print(f"Best grasp candidate:", best_grasp)
    else:
        print(f"No suitable grasp candidates found")
        grasp_candidates.append(None)


Best grasp candidate: (array([0.29514943, 0.27057323, 0.6746725 ]), array([0.29641883, 0.21963018, 0.68000417]), array([-2.63339468e-01,  9.64703054e-01, -5.84202567e-04]), array([ 0.17513741, -0.97968386,  0.09770578]), 172.42870939907812, 0.05123701724287169, 164.9618134008775, 8.671454623809307, 0.022915338271011415)


In [19]:
if compute_grasp_points:
    best_grasps_np = np.array(best_grasps, dtype=object)
    # grasp_candidates_np = np.array(grasp_candidates, dtype=object)

    np.savez("best_grasps.npz", best_grasps=best_grasps_np)
    # np.savez("grasp_candidates.npz", grasp_candidates=grasp_candidates_np)
else:
    loaded_data = np.load("best_grasps.npz", allow_pickle=True)
    best_grasps = loaded_data['best_grasps']
    # grasp_candidates = loaded_data['grasp_candidates']

In [20]:
for i, best_grasp in enumerate(best_grasps):


    p1 = best_grasp[0]
    p2 = best_grasp[1]
    n1 = best_grasp[2]
    n2 = best_grasp[3]

    # diplay selected points and their normals in open3d
    point_cloud = o3d.geometry.PointCloud()

    point_cloud.points = o3d.utility.Vector3dVector(cropped_pcl)
    point_cloud.colors = o3d.utility.Vector3dVector([[255, 0, 0], [0, 255, 0]])

    normals = np.array([n1, n2])
    point_cloud.normals = o3d.utility.Vector3dVector(normals)

    grasp_visualizations = []

    sphere1 = o3d.geometry.TriangleMesh.create_sphere(radius=0.001)
    sphere1.translate(p1)
    sphere1.paint_uniform_color([1, 0, 0])  

    sphere2 = o3d.geometry.TriangleMesh.create_sphere(radius=0.001)
    sphere2.translate(p2)
    sphere2.paint_uniform_color([0, 1, 0]) 

    # Add spheres to the visualization
    grasp_visualizations.append(sphere1)
    grasp_visualizations.append(sphere2)

    # Create lines to represent normals at each grasp point
    normal_line1 = o3d.geometry.LineSet()
    normal_line2 = o3d.geometry.LineSet()

    # Normal vectors n1 and n2 at points p1 and p2 respectively

    line_points1 = [p1, p1 + 0.05 * n1]  # Line representing normal direction for p1
    line_points2 = [p2, p2 + 0.05 * n2]  # Line representing normal direction for p2
   
    normal_line1.points = o3d.utility.Vector3dVector(line_points1)
    normal_line1.lines = o3d.utility.Vector2iVector([[0, 1]])  # Line from p1 to p1 + normal
    normal_line1.colors = o3d.utility.Vector3dVector([[1, 0, 0]])  # Color red

    normal_line2.points = o3d.utility.Vector3dVector(line_points2)
    normal_line2.lines = o3d.utility.Vector2iVector([[0, 1]])  # Line from p2 to p2 + normal
    normal_line2.colors = o3d.utility.Vector3dVector([[0, 1, 0]])  # Color green

    # Add normal lines to the visualization
    grasp_visualizations.append(normal_line1)
    grasp_visualizations.append(normal_line2)

    o3d.visualization.draw_geometries([point_cloud] + grasp_visualizations)

